In [47]:
import duckdb
import pandas as pd
from pathlib import Path

# reads in the statcast data from the database for 2025 season

db_path = Path("~/repos/statcast-database/statcast.duckdb").expanduser()

conn = duckdb.connect(str(db_path))
df = conn.execute("""
    SELECT
        game_pk,
        game_date,
        inning,
        inning_topbot,
        at_bat_number,
        pitch_number,
        pitcher,
        player_name,
        home_team,
        away_team,
        events,
        woba_value,
        woba_denom,
        estimated_woba_using_speedangle
    FROM statcast
    WHERE game_date >= '2025-03-27'
      AND game_date <= '2025-11-30'
    ORDER BY game_pk, at_bat_number, pitch_number
""").df()

df.head()

,game_pk,game_date,inning,inning_topbot,at_bat_number,pitch_number,pitcher,player_name,home_team,away_team,events,woba_value,woba_denom,estimated_woba_using_speedangle
0,776135,2025-09-28,1,Top,1,1,691951,"Aldegheri, Sam",LAA,HOU,NaN,NaN,<NA>,NaN
1,776135,2025-09-28,1,Top,1,2,691951,"Aldegheri, Sam",LAA,HOU,NaN,NaN,<NA>,NaN
2,776135,2025-09-28,1,Top,1,3,691951,"Aldegheri, Sam",LAA,HOU,NaN,NaN,<NA>,NaN
3,776135,2025-09-28,1,Top,1,4,691951,"Aldegheri, Sam",LAA,HOU,NaN,NaN,<NA>,NaN
4,776135,2025-09-28,1,Top,1,5,691951,"Aldegheri, Sam",LAA,HOU,NaN,NaN,<NA>,NaN


In [49]:
# designates the pitching team based on the top or the bottom half of the inning

df["pitching_team"] = df["home_team"].where(
    df["inning_topbot"] == "Top",
    df["away_team"]
)
df.head()

,game_pk,game_date,inning,inning_topbot,at_bat_number,pitch_number,pitcher,player_name,home_team,away_team,events,woba_value,woba_denom,estimated_woba_using_speedangle,pitching_team
0,776135,2025-09-28,1,Top,1,1,691951,"Aldegheri, Sam",LAA,HOU,NaN,NaN,<NA>,NaN,LAA
1,776135,2025-09-28,1,Top,1,2,691951,"Aldegheri, Sam",LAA,HOU,NaN,NaN,<NA>,NaN,LAA
2,776135,2025-09-28,1,Top,1,3,691951,"Aldegheri, Sam",LAA,HOU,NaN,NaN,<NA>,NaN,LAA
3,776135,2025-09-28,1,Top,1,4,691951,"Aldegheri, Sam",LAA,HOU,NaN,NaN,<NA>,NaN,LAA
4,776135,2025-09-28,1,Top,1,5,691951,"Aldegheri, Sam",LAA,HOU,NaN,NaN,<NA>,NaN,LAA


In [50]:

# then groups by the gameID(pk), game_date, pitching_team, pitcherID, and player_name
# it then aggregates globally the pitchers first AB number in the game, last, and the nunmber of pitches

appearances = (
    df.groupby(
        ["game_pk", "game_date", "pitching_team", "pitcher", "player_name"],
        as_index=False
    )
    .agg(
        first_ab=("at_bat_number", "min"),
        last_ab=("at_bat_number", "max"),
        pitches=("pitch_number", "size")
    )
)

In [51]:
appearances

,game_pk,game_date,pitching_team,pitcher,player_name,first_ab,last_ab,pitches
0,776135,2025-09-28,HOU,621121,"McCullers Jr., Lance",8,30,54
1,776135,2025-09-28,HOU,676467,"Gordon, Colton",34,70,68
2,776135,2025-09-28,HOU,681151,"Murray, Jayden",74,80,24
3,776135,2025-09-28,LAA,641401,"Brogdon, Connor",42,51,27
4,776135,2025-09-28,LAA,666171,"Zeferjahn, Ryan",56,60,21
...,...,...,...,...,...,...,...,...
21318,813074,2025-09-30,NYY,596133,"Weaver, Luke",46,48,15
21319,813074,2025-09-30,NYY,608331,"Fried, Max",1,45,102
21320,813074,2025-09-30,NYY,642207,"Williams, Devin",55,58,11
21321,813074,2025-09-30,NYY,657612,"Hill, Tim",67,67,4


In [52]:
# just sorts the df by the player who entered the game first so that we can later add previous pitcher

appearances = appearances.sort_values(
    ["game_pk", "pitching_team", "first_ab"]
)
appearances.head()

,game_pk,game_date,pitching_team,pitcher,player_name,first_ab,last_ab,pitches
0,776135,2025-09-28,HOU,621121,"McCullers Jr., Lance",8,30,54
1,776135,2025-09-28,HOU,676467,"Gordon, Colton",34,70,68
2,776135,2025-09-28,HOU,681151,"Murray, Jayden",74,80,24
6,776135,2025-09-28,LAA,691951,"Aldegheri, Sam",1,41,96
3,776135,2025-09-28,LAA,641401,"Brogdon, Connor",42,51,27


In [53]:
# groups by game and pitching team and then shifts the previous pitcher value down 1 as the previous pitcher

appearances["previous_pitcher"] = (
    appearances
    .groupby(["game_pk", "pitching_team"])["pitcher"]
    .shift(1)
)

appearances["previous_pitcher_name"] = (
    appearances
    .groupby(["game_pk", "pitching_team"])["player_name"]
    .shift(1)
)

appearances.head()

,game_pk,game_date,pitching_team,pitcher,player_name,first_ab,last_ab,pitches,previous_pitcher,previous_pitcher_name
0,776135,2025-09-28,HOU,621121,"McCullers Jr., Lance",8,30,54,NaN,NaN
1,776135,2025-09-28,HOU,676467,"Gordon, Colton",34,70,68,621121.0,"McCullers Jr., Lance"
2,776135,2025-09-28,HOU,681151,"Murray, Jayden",74,80,24,676467.0,"Gordon, Colton"
6,776135,2025-09-28,LAA,691951,"Aldegheri, Sam",1,41,96,NaN,NaN
3,776135,2025-09-28,LAA,641401,"Brogdon, Connor",42,51,27,691951.0,"Aldegheri, Sam"


In [54]:
# groups by game, pitching_team, and pitcher and then gets unique AB #'s 
# to determine the amount of at bats a pitcher has thrown the pitch that results in the final outcome
# filtered out events that were truncated_pa like caught stealing, pick off

completed_pa = df[
    df["events"].notna() &
    (df["events"] != "truncated_pa")
].copy()

pa_counts = (
    completed_pa
    .groupby(["game_pk", "pitching_team", "pitcher"])
    .size()
    .reset_index(name="completed_pa")
)

appearances = appearances.merge(
    pa_counts,
    on=["game_pk", "pitching_team", "pitcher"],
    how="left"
)

appearances["completed_pa"] = appearances["completed_pa"].fillna(0).astype(int)


In [55]:
appearances.head()

,game_pk,game_date,pitching_team,pitcher,player_name,first_ab,last_ab,pitches,previous_pitcher,previous_pitcher_name,completed_pa
0,776135,2025-09-28,HOU,621121,"McCullers Jr., Lance",8,30,54,NaN,NaN,13
1,776135,2025-09-28,HOU,676467,"Gordon, Colton",34,70,68,621121.0,"McCullers Jr., Lance",16
2,776135,2025-09-28,HOU,681151,"Murray, Jayden",74,80,24,676467.0,"Gordon, Colton",7
3,776135,2025-09-28,LAA,691951,"Aldegheri, Sam",1,41,96,NaN,NaN,25
4,776135,2025-09-28,LAA,641401,"Brogdon, Connor",42,51,27,691951.0,"Aldegheri, Sam",6


In [ ]:
# we can see that there are no xwoba values for catcher_interf and there are some missing for doubles, etc.

completed_pa[
    [
        "events",
        "estimated_woba_using_speedangle",
        "woba_value",
        "woba_denom"
    ]
].groupby("events").agg(
    n=("events", "size"),
    xwoba_non_null=("estimated_woba_using_speedangle", "count"),
    woba_non_null=("woba_value", "count"),
    woba_denom_non_null=("woba_denom", "count")
)

,n,xwoba_non_null,woba_non_null,woba_denom_non_null
events,,,,
catcher_interf,86,0,86,86
double,7875,7870,7875,7870
double_play,385,385,385,385
field_error,1029,1021,1029,1021
field_out,75693,75374,75693,75374
fielders_choice,393,390,393,390
fielders_choice_out,334,333,334,333
force_out,3426,3406,3426,3406
grounded_into_double_play,3164,3145,3164,3145


In [58]:
completed_pa[
    completed_pa["estimated_woba_using_speedangle"].isna()
][
    [
        "events",
        "woba_value",
        "woba_denom",
        "estimated_woba_using_speedangle"
    ]
].value_counts(dropna=False)

events                     woba_value  woba_denom  estimated_woba_using_speedangle
intent_walk                0.40        0           NaN                                594
sac_bunt                   0.20        0           NaN                                515
field_out                  0.00        <NA>        NaN                                319
catcher_interf             0.70        1           NaN                                 86
single                     0.90        <NA>        NaN                                 67
sac_bunt                   0.90        0           NaN                                 56
force_out                  0.00        <NA>        NaN                                 20
grounded_into_double_play  0.00        <NA>        NaN                                 19
sac_bunt                   0.20        <NA>        NaN                                 11
field_error                0.90        <NA>        NaN                                  8
double           

In [ ]:
# xwoba_pa is just the plate appearances where there is a valid xwoba

xwoba_pa = completed_pa[
    completed_pa["estimated_woba_using_speedangle"].notna()
].copy()

xwoba_pa["pa_xwoba"] = xwoba_pa[
    "estimated_woba_using_speedangle"
]

In [ ]:
# appearance_xwoba is then just the sum(xwoba for every PA) / plate appearances where there is a valid xwoba
# sum(pa_xwoba)/xwoba_pa

appearance_xwoba = (
    xwoba_pa
    .groupby(["game_pk", "pitching_team", "pitcher"])
    .agg(
        appearance_xwoba=("pa_xwoba", "mean"),
        xwoba_pa=("pa_xwoba", "size")
    )
    .reset_index()
)

In [61]:
appearances = appearances.merge(
    appearance_xwoba,
    on=["game_pk", "pitching_team", "pitcher"],
    how="left"
)

In [62]:
appearances

,game_pk,game_date,pitching_team,pitcher,player_name,first_ab,last_ab,pitches,previous_pitcher,previous_pitcher_name,completed_pa,appearance_xwoba,xwoba_pa
0,776135,2025-09-28,HOU,621121,"McCullers Jr., Lance",8,30,54,NaN,NaN,13,0.393269,13.0
1,776135,2025-09-28,HOU,676467,"Gordon, Colton",34,70,68,621121.0,"McCullers Jr., Lance",16,0.202785,16.0
2,776135,2025-09-28,HOU,681151,"Murray, Jayden",74,80,24,676467.0,"Gordon, Colton",7,0.355000,7.0
3,776135,2025-09-28,LAA,691951,"Aldegheri, Sam",1,41,96,NaN,NaN,25,0.407000,25.0
4,776135,2025-09-28,LAA,641401,"Brogdon, Connor",42,51,27,691951.0,"Aldegheri, Sam",6,0.558248,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21318,813074,2025-09-30,NYY,596133,"Weaver, Luke",46,48,15,608331.0,"Fried, Max",3,0.603499,3.0
21319,813074,2025-09-30,NYY,518585,"Cruz, Fernando",49,51,14,596133.0,"Weaver, Luke",3,0.366166,3.0
21320,813074,2025-09-30,NYY,642207,"Williams, Devin",55,58,11,518585.0,"Cruz, Fernando",4,0.216124,4.0
21321,813074,2025-09-30,NYY,670280,"Bednar, David",63,66,15,642207.0,"Williams, Devin",4,0.297250,4.0


In [63]:
df[(df["game_pk"] == 813074) & (df["pitcher"] == 596133)]

,game_pk,game_date,inning,inning_topbot,at_bat_number,pitch_number,pitcher,player_name,home_team,away_team,events,woba_value,woba_denom,estimated_woba_using_speedangle,pitching_team
726036,813074,2025-09-30,7,Top,46,1,596133,"Weaver, Luke",NYY,BOS,NaN,NaN,<NA>,NaN,NYY
726037,813074,2025-09-30,7,Top,46,2,596133,"Weaver, Luke",NYY,BOS,NaN,NaN,<NA>,NaN,NYY
726038,813074,2025-09-30,7,Top,46,3,596133,"Weaver, Luke",NYY,BOS,NaN,NaN,<NA>,NaN,NYY
726039,813074,2025-09-30,7,Top,46,4,596133,"Weaver, Luke",NYY,BOS,NaN,NaN,<NA>,NaN,NYY
726040,813074,2025-09-30,7,Top,46,5,596133,"Weaver, Luke",NYY,BOS,NaN,NaN,<NA>,NaN,NYY
726041,813074,2025-09-30,7,Top,46,6,596133,"Weaver, Luke",NYY,BOS,NaN,NaN,<NA>,NaN,NYY
726042,813074,2025-09-30,7,Top,46,7,596133,"Weaver, Luke",NYY,BOS,NaN,NaN,<NA>,NaN,NYY
726043,813074,2025-09-30,7,Top,46,8,596133,"Weaver, Luke",NYY,BOS,NaN,NaN,<NA>,NaN,NYY
726044,813074,2025-09-30,7,Top,46,9,596133,"Weaver, Luke",NYY,BOS,NaN,NaN,<NA>,NaN,NYY
726045,813074,2025-09-30,7,Top,46,10,596133,"Weaver, Luke",NYY,BOS,NaN,NaN,<NA>,NaN,NYY
